In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

print("Librerías de ML listas.")

df = pd.read_csv("dataset_inmobiliario_acustico_limpio.csv")
print(f"Dataset cargado con {len(df)} propiedades listas para analizar.")

In [ ]:
# Convertimos la columna de texto 'barrio_1' en múltiples columnas de ceros y unos
df_ml = pd.get_dummies(df, columns=['barrio_1'], drop_first=True)

# variables predictoras (X), queremos adivinar (y)
# Sacamos los precios de la X porque si no, el modelo haría trampa
X = df_ml.drop(['preciousd', 'precio_x_m2'], axis=1)
y = df_ml['precio_x_m2'] # objetivo: predecir el valor del m2

print("Variables estructuradas. Total de características a evaluar:", len(X.columns))

In [ ]:
# 80% para estudiar, 20% para el examen final
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Entrenando el Bosque Aleatorio (esto puede tardar unos segundos)...")
modelo = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)

print("¡Entrenamiento completado!")

In [ ]:
# Le pedimos que adivine los precios de los departamentos que no conoce
predicciones = modelo.predict(X_test)

# Margen de error
error_promedio = mean_absolute_error(y_test, predicciones)
precision_r2 = r2_score(y_test, predicciones)

print(f"Error Absoluto Medio: El modelo se equivoca por un promedio de {error_promedio:.2f} USD por metro cuadrado.")
print(f"Score R2 (Precisión): Explica el {precision_r2 * 100:.2f}% de la variación de los precios.")
print("-" * 50)

# ¿El ruido importa?
importancias = pd.DataFrame({
    'Variable': X.columns,
    'Importancia (%)': modelo.feature_importances_ * 100
}).sort_values(by='Importancia (%)', ascending=False)

print("Top 5 factores que más impactan en el precio del m2:")
print(importancias.head(5).to_string(index=False))

In [ ]:
# ==========================================
# El impacto real del Ruido y Gráfico
# ==========================================

importancias['Variable'] = importancias['Variable'].str.replace('barrio_1_', '')

# Buscamos exactamente el impacto de la contaminación acústica
impacto_ruido = importancias[importancias['Variable'] == 'db_lo_num']
porcentaje_ruido = impacto_ruido['Importancia (%)'].values[0]

print("🔍 ANÁLISIS DEL FACTOR ACÚSTICO:")
print(f"El nivel de ruido (db_lo_num) explica exactamente el {porcentaje_ruido:.2f}% de la variación en el precio del m².")
print("-" * 50)

# Generamos un gráfico visual con el Top 10 para ver dónde queda parado el ruido
top_10 = importancias.head(10)

plt.figure(figsize=(10, 6))
# Invertimos el orden ([::-1]) para que el más importante quede arriba en el gráfico
plt.barh(top_10['Variable'][::-1], top_10['Importancia (%)'][::-1], color='#451c92')
plt.title('Top 10 Variables que definen el Precio del m² en CABA (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importancia Relativa (%)', fontsize=12)
plt.ylabel('Variable', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()